In [2]:
import openai, os
from langchain.prompts import PromptTemplate
from langchain_openai import OpenAI
from langchain.chains import LLMChain

openai.api_key = os.environ.get("OPENAI_API_KEY")
COMPLETION_MODEL = "gpt-3.5-turbo-instruct"
# llm = OpenAI(base_url= "https://api.zhiyungpt.com/v1", model_name=COMPLETION_MODEL, api_key=openai.api_key, max_tokens=2048, temperature=0.5)
llm = OpenAI(model_name=COMPLETION_MODEL, api_key=openai.api_key, max_tokens=2048, temperature=0.5)
multiply_prompt = PromptTemplate(template="请计算一下{question}是多少?", input_variables=["question"])
math_chain = LLMChain(llm=llm, prompt=multiply_prompt, output_key="answer")
answer = math_chain.run({"question": "352乘以493"})
print("OpenAI API 说答案是:", answer)
python_answer = 352 * 493
print("Python 说答案是:", python_answer)

OpenAI API 说答案是: 

352乘以493等于173536。 
Python 说答案是: 173536


In [2]:
multiply_by_python_prompt = PromptTemplate(template="请写一段Python代码，计算{question}?", input_variables=["question"])
math_chain = LLMChain(llm=llm, prompt=multiply_by_python_prompt, output_key="answer")
answer = math_chain.run({"question": "352乘以493"})
print(answer)



result = 352 * 493
print(result)  # 输出结果为173536


In [3]:
multiply_by_python_prompt = PromptTemplate(template="请写一段Python代码，计算{question}?", input_variables=["question"])
math_chain = LLMChain(llm=llm, prompt=multiply_by_python_prompt, output_key="answer")
answer_code = math_chain.run({"question": "352乘以493"})
from langchain.utilities import PythonREPL
python_repl = PythonREPL()
result = python_repl.run(answer_code)
print(result)

Python REPL can execute arbitrary code. Use with caution.


173536



In [4]:
from langchain import LLMMathChain
llm_math = LLMMathChain(llm=llm, verbose=True)
result = llm_math.run("请计算一下352乘以493是多少?")
print(result)

/usr/local/anaconda3/lib/python3.11/site-packages/langchain/chains/llm_math/base.py:57: UserWarning: Directly instantiating an LLMMathChain with an llm is deprecated. Please instantiate with llm_chain argument or using the from_llm class method.
  warnings.warn(




> Entering new LLMMathChain chain...
请计算一下352乘以493是多少?```text
352 * 493
```
...numexpr.evaluate("352 * 493")...

Answer: 173536
> Finished chain.
Answer: 173536


In [9]:
from langchain.chains import LLMRequestsChain
template = """在 >>> 和 <<< 直接是来自Google的原始搜索结果.
请把对于问题 '{query}' 的答案从里面提取出来，如果里面没有相关信息的话就说 "找不到"
请使用如下格式：
Extracted:<answer or "找不到">
>>> {requests_result} <<<
Extracted:"""
PROMPT = PromptTemplate(
    input_variables=["query", "requests_result"],
    template=template,
)
requests_chain = LLMRequestsChain(llm_chain = LLMChain(llm=OpenAI(temperature=0), prompt=PROMPT))
question = "今天上海的天气怎么样？"
inputs = {
    "query": question,
    "url": "https://www.google.com/search?q=" + question.replace(" ", "+")
}
result=requests_chain(inputs)
print(result)
print(result['output'])

{'query': '今天上海的天气怎么样？', 'url': 'https://www.google.com/search?q=今天上海的天气怎么样？', 'output': '今天上海的天气怎么样？ - Google 搜索如果您在几秒钟内没有被重定向，请点击此处。无障碍功能链接跳到主要内容 关闭“连续滚动”开启“连续滚动”无障碍功能帮助无障碍功能反馈  按 / 即可跳到搜索框今天上海的天气怎么样？             选择您要针对哪个元素提供反馈删除查看更多删除查看更多举报不当的联想查询     过滤条件和主题全部图片新闻购物视频更多 已复制到剪贴板找到约 43,100,000 条结果 （用时 0.25 秒）\xa0  搜索结果 · 中国上海市\xa0∙ 选择区域搜索结果天气情况搜索结果 7725°F°F°C °C降水概率：60%湿度：66%'}
今天上海的天气怎么样？ - Google 搜索如果您在几秒钟内没有被重定向，请点击此处。无障碍功能链接跳到主要内容 关闭“连续滚动”开启“连续滚动”无障碍功能帮助无障碍功能反馈  按 / 即可跳到搜索框今天上海的天气怎么样？             选择您要针对哪个元素提供反馈删除查看更多删除查看更多举报不当的联想查询     过滤条件和主题全部图片新闻购物视频更多 已复制到剪贴板找到约 43,100,000 条结果 （用时 0.25 秒）   搜索结果 · 中国上海市 ∙ 选择区域搜索结果天气情况搜索结果 7725°F°F°C °C降水概率：60%湿度：66%


In [5]:
import re
def parse_weather_info(weather_info: str) -> dict:
    # 将天气信息拆分成不同部分
    parts = weather_info.split('; ')
    # 解析天气
    weather = parts[0].strip()
    # 解析温度范围，并提取最小和最大温度
    temperature_range = parts[1].strip().replace('℃', '').split('～')
    temperature_min = int(temperature_range[0])
    temperature_max = int(temperature_range[1])
    # 解析风向和风力
    wind_parts = parts[2].split(' ')
    wind_direction = wind_parts[0].strip()
    wind_force = wind_parts[1].strip()
    # 返回解析后的天气信息字典
    weather_dict = {
        'weather': weather,
        'temperature_min': temperature_min,
        'temperature_max': temperature_max,
        'wind_direction': wind_direction,
        'wind_force': wind_force
    }
    return weather_dict
# 示例
weather_info = "小雨; 10℃～15℃; 东北风 风力4-5级"
weather_dict = parse_weather_info(weather_info)
print(weather_dict)

{'weather': '小雨', 'temperature_min': 10, 'temperature_max': 15, 'wind_direction': '东北风', 'wind_force': '风力4-5级'}


In [8]:
from langchain.chains import TransformChain, SequentialChain
def transform_func(inputs: dict) -> dict:
    text = inputs["output"]
    return {"weather_info" : parse_weather_info(text)}
transformation_chain = TransformChain(input_variables=["output"], 
                                      output_variables=["weather_info"], transform=transform_func)
final_chain = SequentialChain(chains=[requests_chain, transformation_chain], 
                              input_variables=["query", "url"], output_variables=["weather_info"])
final_result = final_chain.run(inputs)
print(final_result)

IndexError: list index out of range

In [14]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import SpacyTextSplitter
from langchain import VectorDBQA
from langchain.document_loaders import TextLoader
llm = OpenAI(temperature=0)
loader = TextLoader('./data/ecommerce_faq.txt')
documents = loader.load()
text_splitter = SpacyTextSplitter(chunk_size=256, pipeline="zh_core_web_sm")
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings()
docsearch = FAISS.from_documents(texts, embeddings)
faq_chain = VectorDBQA.from_chain_type(llm=llm, vectorstore=docsearch, verbose=True)

/usr/local/anaconda3/lib/python3.11/site-packages/langchain/chains/retrieval_qa/base.py:256: UserWarning: `VectorDBQA` is deprecated - please use `from langchain.chains import RetrievalQA`
  warnings.warn(


In [11]:
question = "请问你们的货，能送到三亚吗？大概需要几天？"
result = faq_chain.run(question)
print(result)



> Entering new VectorDBQA chain...

> Finished chain.
 我们支持海南省的配送，包括三亚。一般情况下，大部分城市的订单在2-3个工作日内送达，偏远地区可能需要5-7个工作日。具体的送货时间可能因订单商品、配送地址和物流公司而异。


In [12]:
question = "请问你们的退货政策是怎么样的？" 
result = faq_chain.run(question)
print(result)



> Entering new VectorDBQA chain...

> Finished chain.
 自收到商品之日起7天内，如产品未使用、包装完好，您可以申请退货。某些特殊商品可能不支持退货，请在购买前查看商品详情页面的退货政策。
